In [1]:
#zainstalowanie bibliotek
import pandas as pd

In [2]:
#zimportowanie danych
dane = pd.read_csv("OnlineRetail.csv", encoding='ISO-8859-1')
dane.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [3]:
#wyświetlanie danych na temat zbioru
dane.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [4]:
#zmiana kalumny InvoiceDate na format datetime , wyczyszczenie pustych wierszy oraz stworzenie nowej kolumny TotalPrice
dane["InvoiceDate"] = pd.to_datetime(dane["InvoiceDate"])
dane.dropna(subset=["CustomerID"], inplace=True)
#Usunięcie zwrotów
dane = dane[dane["Quantity"] > 0]
dane = dane[dane["UnitPrice"] > 0]
dane["TotalPrice"] = dane["Quantity"] * dane["UnitPrice"]

In [5]:
import datetime as dt
#ustalenie dnia po ostatniej transakcji w zbiorze
dzien = dane["InvoiceDate"].max() + dt.timedelta(days=1)
rfm = dane.groupby("CustomerID").agg({
    'InvoiceDate': lambda x: (dzien - x.max()).days, 
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'})
#zmienaimy nazwy kolumn
rfm.columns = ["Recency","Frequency","Monetary"]

In [6]:
#przyznanie scoringu
#Recency: 5 to najlepsi (najkrótszy czas), 1 to najgorsi
rfm['R_punktacja'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])

#Frequency: 1 to najrzadsi, 5 to najczęstsi
rfm['F_punktacja'] = pd.qcut(rfm['Frequency'].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])

#Monetary: 1 to najmniej wydający, 5 to najwięcej
rfm['M_punktacja'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

In [7]:
#Łączenie wyników 
rfm["RFM_punktacja"]= rfm['R_punktacja'].astype(str) + rfm['F_punktacja'].astype(str) + rfm['M_punktacja'].astype(str)
#Stworzenie słownika 
słownik = {
    r'[1-2][1-2]': 'Uśpieni',                 # Kupowali dawno i rzadko.
    r'[1-2][3-4]': 'Zagrożeni',               # Dawniej lojalni, ale dawno ich nie było
    r'[1-2]5': 'Nie można ich stracić',      # Najbardziej lojalni, którzy nagle zniknęli
    r'3[1-2]': 'Zasypiający',                # Średnia aktualność i niska częstotliwość
    r'33': 'Wymagający uwagi',               # Średniacy – ani dobrzy, ani źli
    r'[3-4][4-5]': 'Lojalni klienci',         # Regularnie kupujący, filar firmy
    r'41': 'Obiecujący',                     # Kupili niedawno, ale to ich początki
    r'51': 'Nowi klienci',                   # Świeżo pozyskani klienci
    r'[4-5][2-3]': 'Potencjalni lojaliści',   # Bardzo obiecujący, kupili kilka razy niedawno
    r'5[4-5]': 'Mistrzowie' }                 # Najlepsi z najlepszych
#Stworzenie kolumny Segment oraz użycie nazw z słownika
rfm["Segment"] = rfm['R_punktacja'].astype(str) + rfm['F_punktacja'].astype(str)
rfm['Segment'] = rfm['Segment'].replace(słownik, regex=True)
#Podsumowanie segmentów
podsumowanie_segmentów = rfm['Segment'].value_counts()
podsumowanie_segmentów.head()

Segment
Uśpieni                  1065
Lojalni klienci           827
Mistrzowie                633
Zagrożeni                 580
Potencjalni lojaliści     492
Name: count, dtype: int64

In [8]:
#Zapisanie danych w Excelu
rfm.to_excel("Wyniki_RFM_E-commerce.xlsx")